# MELD quickstart

MELD fuses tile-level **UNI-v2** (1536-d) and **Virchow2** (1280-d) embeddings into a compact
per-tile representation. This notebook is a minimal, end-to-end example covering:

1. Creating the architecture from scratch
2. Loading a pretrained checkpoint (MELD-COMET or MELD-CCDI)
3. Running inference (feature extraction) on tile embeddings
4. Fine-tuning a loaded model on your own data
5. Training a model from scratch

**Note on your data:** MELD expects one row per tile with shape `(N_TILES, 8448)`, where the
8448 columns are `[virchow2_a, virchow2_b, virchow2_c, uni_v2_a, uni_v2_b, uni_v2_c]`
(1280+1280+1280+1536+1536+1536). The "3 copies" reflect how tiles were cached for
COMET/CCDI/COG (e.g. multiple crops/scales per tile) during training. If your extraction
pipeline differs, adapt the small helper in Step 3, or simply repeat a single embedding 3x —
then fine-tune (Step 4) to adapt the model to your feature distribution.

In [ ]:
# If you installed this repo as a package (`pip install -e .`), this is all you need:
import numpy as np

import meld

print("MELD input dim:", meld.INPUT_DIM, " | MELD feature dim:", meld.FEATURE_DIM)

## 1. Create the architecture

This builds the network with random weights — useful if you want to inspect the graph, or
train from scratch (Step 5).

In [ ]:
model = meld.create_architecture()
model.summary()

## 2. Load a pretrained checkpoint

Two checkpoints are released, both trained for 20 epochs:
- `"comet"` — MELD-COMET, distilled/trained on the COMET dataset
- `"ccdi"` — MELD-CCDI, distilled/trained on the CCDI dataset

Weights are shipped as `.weights.h5` (weights-only, no optimizer state) under `weights/`,
which keeps the release small; if you cloned this repo without the large files, download them
from the Hugging Face model repo linked in the README and place them in `weights/`, or pass an
explicit path to `load_model(weights=<path>)`.

In [ ]:
meld_comet = meld.load_model("comet")   # or meld.load_model("ccdi"), or a custom path
meld_comet = meld.set_inference_mode(meld_comet)
print("Loaded MELD-COMET with", meld_comet.count_params(), "parameters.")

## 3. Run inference / extract MELD features

Replace the synthetic `tile_features` array below with your own tiles' concatenated
Virchow2 + UNI-v2 embeddings (shape `(n_tiles, 8448)`; see the note at the top of this
notebook). `extract_features` returns the 3000-d normalized MELD embedding per tile, which
is what downstream models (e.g. tumor/subtype classifiers) are trained on.

In [ ]:
# --- Replace this block with your real tile features ---
n_tiles = 8
virchow2 = np.random.randn(n_tiles, 3, 1280).astype("float32")  # e.g. 3 crops/scales per tile
uni_v2 = np.random.randn(n_tiles, 3, 1536).astype("float32")
tile_features = np.concatenate(
    [virchow2.reshape(n_tiles, -1), uni_v2.reshape(n_tiles, -1)], axis=1
)  # -> (n_tiles, 8448)
# ---------------------------------------------------------

meld_features = meld.extract_features(meld_comet, tile_features)
print("MELD embedding shape:", meld_features.shape)  # (n_tiles, 3000)

## 4. Fine-tune a loaded model on your own data

`meld.finetune` continues training an already-loaded model with a small learning rate. Any
object accepted by `keras.Model.fit` works for `data` — a `tf.data.Dataset`, a
`keras.utils.Sequence`, or a plain `(X, X)` numpy pair. Input **and** target are the same
`(batch, 8448)` tile-feature array; the loss function only needs the raw input as target.

Below is the smallest possible example using an in-memory `tf.data.Dataset`. For real
training, swap this out for your own generator that reads cached tile features from disk.

In [ ]:
import tensorflow as tf

# --- Replace with your own tile features, ideally thousands+ of tiles ---
X_train = np.random.randn(512, meld.INPUT_DIM).astype("float32")
# --------------------------------------------------------------------

train_ds = tf.data.Dataset.from_tensor_slices((X_train, X_train)).shuffle(512).batch(32)

meld_comet_finetuned = meld.load_model("comet")  # start from the pretrained checkpoint
meld_comet_finetuned = meld.finetune(meld_comet_finetuned, train_ds, epochs=1)

## 5. Train a model from scratch

`meld.train` is the same call, just starting from a freshly created (randomly initialized)
architecture instead of pretrained weights.

In [ ]:
new_model = meld.create_architecture()
new_model = meld.train(new_model, train_ds, epochs=1)

# Save just the weights (small, portable) for later reuse with meld.load_model:
new_model.save_weights("my_meld_model.weights.h5")